# A1 – Calidad de los datos

En este notebook revisamos los datos antes de analizarlos. Primero los miramos tal como vinieron (`cargar_crudo`). Al final comprobamos que la versión limpia (`cargar_limpio`) aplica lo que decidimos en `DECISIONES.md`.

1. Cómo es cada tabla
2. Cruces entre tablas
3. Los tres problemas más importantes
4. Métricas de calidad de cada tabla
5. ¿Los datos dicen lo mismo que el enunciado?
6. Cómo se relacionan las tablas
7. Comprobación de los datos limpios

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
from src.carga import cargar_crudo, cargar_limpio, RAW, FECHAS

d = cargar_crudo()
cli = d["clientes"]
trx = d["transacciones"]
prd = d["productos"]
dev = d["devoluciones"]
crm = d["interacciones_crm"]
cam = d["campanas_marketing"]

for nombre, tabla in d.items():
    print(f"{nombre:20s} {tabla.shape[0]:6d} filas  {tabla.shape[1]:2d} columnas")

## 1. Cómo es cada tabla

Leemos cada archivo como texto, sin convertir nada, para ver los valores exactamente como vienen. En cada columna buscamos:

- celdas vacías,
- datos faltantes escritos como texto ("null", "N/A", "No especificado"…),
- filas repetidas,
- valores mínimos y máximos,
- fechas que no se puedan leer.

In [ ]:
NULOS_DISFRAZADOS = ["nan", "null", "none", "n/a", "na", "-", "?", "no especificado", "desconocido"]

for nombre in d:
    t = pd.read_csv(RAW / f"{nombre}.csv", dtype=str, keep_default_na=False)
    id_col = t.columns[0]
    print(f"\n=== {nombre}: {len(t)} filas")
    print(f"duplicados exactos: {t.duplicated().sum()}  |  {id_col} repetidos: {t[id_col].duplicated().sum()}")

    for col in t.columns:
        s = t[col].str.strip()
        vacios = (s == "").sum()
        disfrazados = s.str.lower().isin(NULOS_DISFRAZADOS).sum()
        linea = f"  {col:22s} vacíos={vacios:4d}  disfrazados={disfrazados:4d}  únicos={s.nunique():6d}"

        num = pd.to_numeric(s, errors="coerce")
        if num.notna().mean() > 0.9:
            linea += f"  | min={num.min():g}  mediana={num.median():g}  max={num.max():g}"
        elif col in FECHAS.get(nombre, []):
            f = pd.to_datetime(s, errors="coerce", format="%Y-%m-%d")
            linea += f"  | {f.min().date()} a {f.max().date()}  (no se pueden leer: {f.isna().sum() - vacios})"
        elif s.nunique() <= 10:
            linea += f"  | {s.value_counts().to_dict()}"
        print(linea)

**Resultado:**
- No hay celdas vacías ni filas repetidas en ninguna tabla.
- El único dato faltante escrito como texto es el género "No especificado", en 433 clientes.
- Todas las fechas van del 1 de enero de 2023 al 31 de diciembre de 2024 (24 meses).
- Hay 318 productos, no 320 como dice el enunciado.

## 2. Cruces entre tablas

Comprobamos lo que pide el enunciado (devoluciones sin una venta de origen y fechas que no tienen sentido) y además que los precios y montos cuadren.

In [ ]:
def mostrar(texto, valor):
    print(f"{texto:62s} {valor}")

print("--- Claves huérfanas")
mostrar("transacciones con cliente inexistente", (~trx.cliente_id.isin(cli.cliente_id)).sum())
mostrar("transacciones con producto inexistente", (~trx.producto_id.isin(prd.producto_id)).sum())
mostrar("devoluciones sin transacción de origen", (~dev.transaccion_id.isin(trx.transaccion_id)).sum())
for nombre, t in [("devoluciones", dev), ("crm", crm), ("campañas", cam)]:
    mostrar(f"{nombre} con cliente inexistente", (~t.cliente_id.isin(cli.cliente_id)).sum())
mostrar("clientes sin ninguna transacción", (~cli.cliente_id.isin(trx.cliente_id)).sum())

print("\n--- Transacciones")
calculado = (trx.cantidad * trx.precio_unitario * (1 - trx.descuento)).round(2)
mostrar("monto_neto distinto de cantidad*precio*(1-descuento)", ((trx.monto_neto - calculado).abs() > 0.05).sum())
m = trx.merge(prd[["producto_id", "precio_lista", "costo", "categoria"]], on="producto_id")
mostrar("precio_unitario distinto del precio de lista", ((m.precio_unitario - m.precio_lista).abs() > 0.01).sum())
mostrar("venta por unidad por debajo del costo", (m.monto_neto / m.cantidad < m.costo).sum())
con_registro = trx.merge(cli[["cliente_id", "fecha_registro"]], on="cliente_id", how="left")
mostrar("compra anterior al registro del cliente", (con_registro.fecha < con_registro.fecha_registro).sum())

print("\n--- Productos")
margen_calc = ((prd.precio_lista - prd.costo) / prd.precio_lista * 100).round(1)
mostrar("margen_bruto_pct que no cuadra con costo y precio", ((margen_calc - prd.margen_bruto_pct).abs() > 0.15).sum())

print("\n--- Devoluciones")
dd = dev.merge(trx[["transaccion_id", "cliente_id", "fecha", "monto_neto"]],
               on="transaccion_id", how="left", suffixes=("", "_trx"))
mostrar("devolución antes de la compra", (dd.fecha_devolucion < dd.fecha).sum())
mostrar("se devolvió más de lo que se pagó", (dd.monto_devuelto > dd.monto_neto + 0.01).sum())
mostrar("el cliente de la devolución no es el de la compra", (dd.cliente_id != dd.cliente_id_trx).sum())
mostrar("días de la compra a la devolución (mediana / p95 / máx)",
        (dd.fecha_devolucion - dd.fecha).dt.days.quantile([0.5, 0.95, 1]).tolist())
mostrar("% de transacciones con devolución", round(len(dev) / len(trx) * 100, 2))

print("\n--- CRM")
q1, q3 = crm.tiempo_respuesta_hrs.quantile([0.25, 0.75])
umbral = q3 + 1.5 * (q3 - q1)
mostrar(f"tiempo de respuesta sobre Q3+1,5·IQR ({umbral:.1f} h)", (crm.tiempo_respuesta_hrs > umbral).sum())
mostrar("tiempo de respuesta de más de una semana", (crm.tiempo_respuesta_hrs > 168).sum())

print("\n--- Campañas")
mostrar("convertido=1 sin haber abierto", ((cam.convertido == 1) & (cam.abierto == 0)).sum())
mostrar("conversión global % / apertura %", f"{cam.convertido.mean():.2%} / {cam.abierto.mean():.2%}")

print("\n--- Segmento declarado vs comportamiento")
ultima = trx.groupby("cliente_id").fecha.max()
c = cli.set_index("cliente_id").join(ultima.rename("ultima"))
recientes = c.ultima >= trx.fecha.max() - pd.Timedelta(days=90)
mostrar("'Inactivo' con compras en los últimos 90 días", ((c.segmento == "Inactivo") & recientes).sum())

## 3. Los tres problemas más importantes

1. Hay compras anteriores a la fecha de registro del cliente. ¿Depende del canal? Por ejemplo: gente que compraba en tienda antes de registrarse en la web.
2. Hay ventas por debajo del costo. ¿De dónde salen?
3. ¿Las campañas marcadas como convertidas tienen una compra que lo confirme? ¿Hay campañas "convertidas" que se enviaron después de la compra?

In [ ]:
con_registro["antes"] = con_registro.fecha < con_registro.fecha_registro
print("1) % de compras antes del registro, por canal:")
print(con_registro.groupby("canal").antes.mean().round(3).to_string())

por_cliente = con_registro.groupby("cliente_id").agg(registro=("fecha_registro", "first"), primera=("fecha", "min"))
correlacion = por_cliente.registro.map(pd.Timestamp.toordinal).corr(por_cliente.primera.map(pd.Timestamp.toordinal))
print("   correlación entre fecha de registro y primera compra:", round(correlacion, 3))

In [ ]:
bajo_costo = m[m.monto_neto / m.cantidad < m.costo]
print("2) Ventas bajo costo:", len(bajo_costo))
print("   descuento aplicado (mín / mediana / máx):", bajo_costo.descuento.quantile([0, 0.5, 1]).round(3).tolist())
print("   por categoría:", bajo_costo.categoria.value_counts().to_dict())
print("   margen de lista de esas categorías:", prd.groupby("categoria").margen_bruto_pct.median().round(1).to_dict())

In [ ]:
convertidas = cam[cam.convertido == 1]
cruce = convertidas.merge(trx[["cliente_id", "fecha"]], on="cliente_id", how="left")
cruce["dias"] = (cruce.fecha - cruce.fecha_envio).dt.days

print(f"3) {len(convertidas)} campañas marcadas como convertidas")
for ventana in (7, 14, 30):
    con_compra = cruce[cruce.dias.between(0, ventana)].campana_id.unique()
    print(f"   con compra en los {ventana:2d} días siguientes: {convertidas.campana_id.isin(con_compra).sum()}")

despues = set(cruce[cruce.dias.between(0, 30)].campana_id)
antes = set(cruce[cruce.dias.between(-30, -1)].campana_id)
print("   solo hay una compra en los 30 días ANTERIORES al envío:", len(antes - despues))

**Resultado** (el detalle está en `DECISIONES.md`):

1. **Compras antes del registro:** pasa igual en los tres canales, y la fecha de registro no tiene relación con la primera compra. La empresa tiene 12 años, pero todos los registros son de 2023 y 2024. Concluimos que esa fecha es el alta en un sistema nuevo, así que no la usamos para medir la antigüedad del cliente.
2. **Ventas bajo costo:** casi todas son de Electrónica, la categoría con menos margen. Con descuentos de entre 9% y 34,5%, la venta termina en pérdida. No es un error: es un hallazgo para A2.
3. **Campañas convertidas:** solo 179 de 797 tienen una compra en los 30 días siguientes. En 102 casos, la única compra cercana es anterior al envío (el caso que menciona el enunciado). La marca de conversión sirve para comparar tasas, pero no para calcular cuánto dinero trajo cada campaña.

## 4. Métricas de calidad de cada tabla

El enunciado pide cuatro métricas. Así las medimos:

- **Completitud:** qué porcentaje de las celdas tiene un valor real (ni vacío ni "No especificado").
- **Unicidad:** qué porcentaje de las filas tiene un código que no se repite.
- **Consistencia:** qué porcentaje de las filas cumple todas las reglas de su tabla. Debajo aparece cuántas filas incumple cada regla.
- **Oportunidad:** qué porcentaje de los registros está al día en la fecha de corte (31 de diciembre de 2024). En clientes, devoluciones y atención al cliente, que la fecha caiga dentro de los 24 meses del estudio. En ventas y campañas, además, que haya pasado suficiente tiempo para que el dato esté completo. Por ejemplo, una compra del 28 de diciembre todavía podía devolverse después del 31, así que no sabemos si se devolvió.

In [ ]:
inicio, fin = pd.Timestamp("2023-01-01"), pd.Timestamp("2024-12-31")

def completitud(nombre):
    t = pd.read_csv(RAW / f"{nombre}.csv", dtype=str, keep_default_na=False)
    faltan = t.apply(lambda s: (s.str.strip() == "") | s.str.strip().str.lower().isin(NULOS_DISFRAZADOS))
    return 100 * (1 - faltan.values.mean())

ult_compra = trx.groupby("cliente_id").fecha.max()
cli_ult = cli.cliente_id.map(ult_compra)
precio_ok = trx.merge(prd[["producto_id", "precio_lista"]], on="producto_id", how="left").set_index(trx.index)

reglas = {
    "clientes": {
        "edad entre 18 y 100": cli.edad.between(18, 100),
        "género informado": cli.genero != "No especificado",
        "'Inactivo' sin compras en los últimos 90 días": ~((cli.segmento == "Inactivo") & (cli_ult >= fin - pd.Timedelta(days=90))),
    },
    "transacciones": {
        "cliente existe": trx.cliente_id.isin(cli.cliente_id),
        "producto existe": trx.producto_id.isin(prd.producto_id),
        "monto_neto cuadra": (trx.monto_neto - calculado).abs() <= 0.05,
        "precio = precio de lista": (precio_ok.precio_unitario - precio_ok.precio_lista).abs() <= 0.01,
        "compra posterior al registro": ~con_registro.set_index(trx.index).antes,
    },
    "productos": {
        "costo < precio de lista": prd.costo < prd.precio_lista,
        "margen_bruto_pct cuadra": (margen_calc - prd.margen_bruto_pct).abs() <= 0.15,
    },
    "devoluciones": {
        "transacción de origen existe": dev.transaccion_id.isin(trx.transaccion_id),
        "mismo cliente que la compra": (dd.cliente_id == dd.cliente_id_trx).values,
        "después de la compra": (dd.fecha_devolucion >= dd.fecha).values,
        "no devuelve más de lo pagado": (dd.monto_devuelto <= dd.monto_neto + 0.01).values,
    },
    "interacciones_crm": {
        "cliente existe": crm.cliente_id.isin(cli.cliente_id),
        "tiempo de respuesta ≥ 0": crm.tiempo_respuesta_hrs >= 0,
        "respuesta en menos de una semana": crm.tiempo_respuesta_hrs <= 168,
    },
    "campanas_marketing": {
        "cliente existe": cam.cliente_id.isin(cli.cliente_id),
        "convertida implica abierta": ~((cam.convertido == 1) & (cam.abierto == 0)),
        "conversión con compra en 30 días": ~((cam.convertido == 1) & ~cam.campana_id.isin(despues)),
    },
}

# Oportunidad: fecha dentro del período y, donde aplica, con tiempo suficiente para que el dato esté completo
p95_devolucion = int((dd.fecha_devolucion - dd.fecha).dt.days.quantile(0.95))
oportunidad = {
    "clientes": cli.fecha_registro.between(inicio, fin),
    "transacciones": trx.fecha <= fin - pd.Timedelta(days=p95_devolucion),
    "productos": pd.Series(True, index=prd.index),  # no tiene fecha
    "devoluciones": dev.fecha_devolucion.between(inicio, fin),
    "interacciones_crm": crm.fecha.between(inicio, fin),
    "campanas_marketing": cam.fecha_envio <= fin - pd.Timedelta(days=30),
}

filas = []
for nombre, t in d.items():
    ok = pd.concat({k: pd.Series(np.asarray(v), index=t.index) for k, v in reglas[nombre].items()}, axis=1)
    filas.append({
        "dataset": nombre,
        "filas": len(t),
        "completitud %": round(completitud(nombre), 2),
        "unicidad %": round(100 * (1 - t.iloc[:, 0].duplicated().mean()), 2),
        "consistencia %": round(100 * ok.all(axis=1).mean(), 2),
        "oportunidad %": round(100 * oportunidad[nombre].mean(), 2),
    })
    print(f"\n{nombre}")
    for regla, cumple in ok.items():
        print(f"   {regla:48s} incumplen: {(~cumple).sum()}")

calidad = pd.DataFrame(filas).set_index("dataset")
calidad

**Cómo leer la tabla:**

- La completitud y la unicidad están casi al 100%. Los datos no tienen problemas de forma; los problemas están en lo que significan.
- La consistencia de ventas baja a 50% por una sola regla: las compras anteriores al registro. Ya explicamos que el problema es la fecha de registro, no las ventas.
- La consistencia de campañas baja por las conversiones que no tienen una compra que las confirme.
- En oportunidad, las ventas de los últimos 29 días cuentan como incompletas, porque todavía podían recibir una devolución. Las campañas del último mes también, porque no pasaron los 30 días para convertir. En la práctica el efecto es pequeño: diciembre tiene 13,4% de devoluciones, igual que el resto del año. Aun así, hay que tenerlo presente al mirar los últimos meses.

In [ ]:
# Comprobación de lo anterior: tasa de devolución por mes de compra
tasa_mes = trx.assign(devuelta=trx.transaccion_id.isin(dev.transaccion_id)).groupby(trx.fecha.dt.to_period("M")).devuelta.mean() * 100
print(f"p95 de días hasta la devolución: {p95_devolucion}")
print(tasa_mes.tail(4).round(1).to_string())
print(f"promedio del resto del período: {tasa_mes.iloc[:-2].mean():.1f}%")

## 5. ¿Los datos dicen lo mismo que el enunciado?

El enunciado da tres cifras como punto de partida del caso. Antes de construir la historia sobre ellas, las comprobamos. Así medimos cada una:

- **Ticket promedio:** cuánto paga un cliente, en promedio, en cada compra. Es el total vendido dividido entre el número de compras. El enunciado dice que bajó 8% "interanual". Lo comparamos de dos formas: 2024 contra 2023 completos, y el último trimestre de 2024 contra el mismo trimestre de 2023.
- **Tasa de devoluciones:** de cada 100 ventas de un trimestre, cuántas se devolvieron. El enunciado dice que "aumentó al 14%" en el último trimestre.
- **Conversión de campañas:** de cada 100 campañas enviadas en un trimestre, cuántas quedaron marcadas como convertidas (el cliente compró). El enunciado dice que bajó de 22% a 11% en 18 meses.

In [ ]:
por_anio = trx.groupby(trx.fecha.dt.year).monto_neto.mean()
print(f"Ticket promedio 2023: {por_anio[2023]:.2f} | 2024: {por_anio[2024]:.2f} | cambio: {100 * (por_anio[2024] / por_anio[2023] - 1):.1f}%")

trx_q = trx.fecha.dt.to_period("Q")
ticket_q = trx.groupby(trx_q).monto_neto.mean()
print(f"Ticket del último trimestre 2023: {ticket_q.iloc[3]:.2f} | 2024: {ticket_q.iloc[-1]:.2f} | cambio: {100 * (ticket_q.iloc[-1] / ticket_q.iloc[3] - 1):.1f}%")

primeros_6 = cam[cam.fecha_envio < "2023-07-01"].convertido.mean() * 100
ultimos_6 = cam[cam.fecha_envio >= "2024-07-01"].convertido.mean() * 100
print(f"Conversión primeros 6 meses: {primeros_6:.1f}% | últimos 6 meses: {ultimos_6:.1f}%")
print(f"Ventas por canal (%): {(trx.canal.value_counts(normalize=True) * 100).round(1).to_dict()}")

tasa_dev = dev.merge(trx[["transaccion_id"]].assign(q=trx_q), on="transaccion_id").groupby("q").size() / trx.groupby(trx_q).size() * 100
conversion = cam.groupby(cam.fecha_envio.dt.to_period("Q")).convertido.mean() * 100
pd.DataFrame({"ticket promedio": ticket_q.round(1), "devoluciones %": tasa_dev.round(1), "conversión %": conversion.round(1)})

**Resultado:** solo una de las tres cifras se parece a lo que dicen los datos, y solo en parte.

| Métrica | Qué dice el enunciado | Qué muestran los datos | ¿Coincide? |
|---|---|---|---|
| Ticket promedio | Bajó 8% en un año | En 2023 cada compra promedió 351 y en 2024, 344: bajó 1,9%. Si comparamos solo el último trimestre de cada año, bajó 0,2% | **No.** El gasto por compra casi no cambió |
| Devoluciones | Subieron hasta 14% en el último trimestre | En el último trimestre fue 13,5%, pero todos los trimestres desde 2023 están entre 13,0% y 13,8% | **A medias.** El nivel se parece, pero no hubo subida: siempre estuvo así |
| Conversión de campañas | Bajó de 22% a 11% en 18 meses | Bajó de 11,3% (primer trimestre de 2023) a 5,6% (último trimestre de 2024) | **En parte.** Se cayó a la mitad, como dice el enunciado, pero empezó en 11%, no en 22% |

Además, hay un canal de venta que el enunciado no menciona: la app móvil, con el 17% de las ventas.

**Qué significa para la historia:** el problema más grave está en el marketing, porque la conversión se cayó a la mitad. El gasto por compra está estable. Las devoluciones son altas (casi 1 de cada 7 ventas), pero no están empeorando.

## 6. Cómo se relacionan las tablas

El diagrama está en `docs/modelo.md`. En resumen:

- **Ventas** es la tabla central. Cada fila es un producto que un cliente compró en una fecha.
- **Clientes, productos y fechas** describen cada venta.
- **Devoluciones** se conecta con una venta concreta.
- **Campañas y atención al cliente** solo se conectan con el cliente, no con una venta. Por eso no sabemos qué campaña generó qué compra y hay que deducirlo por las fechas.

Aquí comprobamos cuántos registros tiene cada relación:

In [ ]:
print("devoluciones por transacción (máx):", dev.transaccion_id.value_counts().max())
print("¿cada devolución apunta a una sola transacción?", dev.devolucion_id.is_unique)
print("transacciones por cliente (mediana / máx):", trx.cliente_id.value_counts().median(), trx.cliente_id.value_counts().max())
print("campañas por cliente (mediana / máx):", cam.cliente_id.value_counts().median(), cam.cliente_id.value_counts().max())
print("interacciones por cliente (mediana / máx):", crm.cliente_id.value_counts().median(), crm.cliente_id.value_counts().max())

## 7. Comprobación de los datos limpios

Comprobamos que los datos limpios tienen las mismas filas que los originales y que las marcas coinciden con lo que contamos arriba. También calculamos las cifras que usamos en `DECISIONES.md` para justificar los umbrales.

In [ ]:
limpio = cargar_limpio(recalcular=True)

for nombre in d:
    assert len(limpio[nombre]) == len(d[nombre]), nombre

t, c, k = limpio["transacciones"], limpio["clientes"], limpio["campanas_marketing"]
print("ventas bajo costo:", t.bajo_costo.sum())
print("outliers de monto por categoría:", t.outlier_cat.sum())
print("clientes sin compras:", c.sin_compras.sum())
print("conversiones respaldadas:", k.conversion_respaldada.sum(), "de", (k.convertido == 1).sum())
print("CRM con respuesta atípica:", limpio["interacciones_crm"].respuesta_atipica.sum())

# Datos que respaldan los umbrales de DECISIONES.md
entre_compras = trx.sort_values("fecha").groupby("cliente_id").fecha.diff().dt.days
print("días entre compras (mediana / p75 / p90):", entre_compras.quantile([0.5, 0.75, 0.9]).tolist())
q1, q3 = trx.monto_neto.quantile([0.25, 0.75])
global_iqr = trx.monto_neto > q3 + 1.5 * (q3 - q1)
print("outliers de monto con IQR global:", global_iqr.sum(), "| de Electrónica:", (global_iqr & t.categoria.eq("Electrónica")).sum())
print("clientes sin compras que reciben campañas:", c[c.sin_compras].cliente_id.isin(k.cliente_id).sum())
conv = k[k.convertido == 1]
print("% de conversiones respaldadas por tipo de campaña:")
print((conv.groupby("tipo_campana").conversion_respaldada.mean() * 100).round(1).to_string())
print("% respaldadas por año:", (conv.groupby(conv.fecha_envio.dt.year).conversion_respaldada.mean() * 100).round(1).to_dict())
print(f"conversión si solo contáramos las respaldadas: {k.conversion_respaldada.mean():.1%}")

# ¿Los que llegan a 180 días sin comprar vuelven? Solo casos con 180 días más de datos para observarlo
orden = trx.sort_values(["cliente_id", "fecha"]).copy()
orden["siguiente"] = orden.groupby("cliente_id").fecha.shift(-1)
orden["dia_180"] = orden.fecha + pd.Timedelta(days=180)
observables = orden[orden.dia_180 <= trx.fecha.max() - pd.Timedelta(days=180)]
llegan_a_180 = observables[observables.siguiente.isna() | (observables.siguiente > observables.dia_180)]
vuelven = llegan_a_180.siguiente <= llegan_a_180.dia_180 + pd.Timedelta(days=180)
print(f"clientes que llegan a 180 días sin comprar: {len(llegan_a_180)} casos; vuelven en los 180 días siguientes: {vuelven.mean():.1%}")
print(f"margen bruto total: {t.margen_bruto.sum():,.0f} sobre ventas de {t.monto_neto.sum():,.0f} ({t.margen_bruto.sum() / t.monto_neto.sum():.1%})")
pd.crosstab(c.segmento, c.segmento_rfm, margins=True)

El 52% de los clientes que llegan a 180 días sin comprar vuelve después. Por eso el segmento calculado "Inactivo" significa "en riesgo", no "perdido".

La tabla del final compara el segmento que trae cada cliente con el que calculamos según cómo compra. Coinciden en líneas generales, pero no del todo: por ejemplo, muchos clientes "Esporádico" y "Nuevo" llevan más de 180 días sin comprar.

En A2 y A3 usamos el segmento original, que es el que pide el enunciado, y el calculado para contrastar.